# 3 models for fixed affects

Yes—for your research question, a one-year lagged fixed-effects model is probably better as the primary analysis.

It tests:

$$ Coverage_{s,t} \rightarrow Cases_{s,t+1} $$

while still controlling for state and year fixed effects:

$$ \log(1+Cases_{s,t+1}) = \beta Coverage_{s,t} +\alpha_s+\gamma_{t+1}+\epsilon_{s,t} $$

Advantages:

Establishes temporal ordering: coverage is measured before cases.
Reduces same-year reverse causation, where an outbreak causes vaccination uptake.
Retains state and year fixed effects to address confounding.
Fits your project’s interest in whether coverage predicts later disease burden.

However, a one-year lag is not automatically biologically “correct.” Annual coverage can affect transmission during the same year, and susceptible populations accumulate over multiple years. Therefore, a defensible strategy is:

- Primary model: one-year lagged fixed effects.
- Sensitivity model: same-year fixed effects.
- Exploratory model: two-year lagged fixed effects.

In [32]:
import pandas as pd

cases_df = pd.read_csv('../app/data/tycho_cases.csv')
cases_df.head()

,year,state,measles_cases,mumps_cases,pertussis_cases
0,1995,AK,NaN,13.0,1.0
1,1995,AL,NaN,4.0,38.0
2,1995,AR,2.0,10.0,41.0
3,1995,AZ,10.0,2.0,151.0
4,1995,CA,108.0,206.0,463.0


In [33]:
coverage_df = pd.read_csv('../app/data/nis_vacc_coverage.csv')
coverage_df.head()

,state,year,measles_coverage_pct,measles_n,measles_n_vaccinated,mumps_coverage_pct,mumps_n,mumps_n_vaccinated,pertussis_coverage_pct,pertussis_n,pertussis_n_vaccinated
0,AK,1995,89.86,194.0,177.0,89.86,194.0,177.0,77.58,194.0,159.0
1,AK,1996,84.55,260.0,225.0,84.55,260.0,225.0,78.25,260.0,210.0
2,AK,1997,87.41,291.0,257.0,87.41,291.0,257.0,80.98,291.0,242.0
3,AK,1998,87.06,34.0,30.0,87.06,34.0,30.0,82.01,34.0,28.0
4,AK,1999,90.67,349.0,321.0,90.67,349.0,321.0,83.54,349.0,296.0


In [34]:
panel_df = pd.merge(cases_df, coverage_df, on=['year', 'state'])
panel_df.head()

,year,state,measles_cases,mumps_cases,pertussis_cases,measles_coverage_pct,measles_n,measles_n_vaccinated,mumps_coverage_pct,mumps_n,mumps_n_vaccinated,pertussis_coverage_pct,pertussis_n,pertussis_n_vaccinated
0,1995,AK,NaN,13.0,1.0,89.86,194.0,177.0,89.86,194.0,177.0,77.58,194.0,159.0
1,1995,AL,NaN,4.0,38.0,88.77,419.0,386.0,88.77,419.0,386.0,78.93,419.0,357.0
2,1995,AR,2.0,10.0,41.0,90.43,237.0,218.0,90.43,237.0,218.0,77.24,237.0,190.0
3,1995,AZ,10.0,2.0,151.0,82.29,372.0,319.0,82.29,372.0,319.0,74.52,372.0,296.0
4,1995,CA,108.0,206.0,463.0,90.32,681.0,627.0,90.32,681.0,627.0,76.76,681.0,542.0


In [35]:
duplicate_rows = panel_df[
    panel_df.duplicated(
        subset=["state", "year"],
        keep=False
    )
]

print("Duplicate state-year rows:", len(duplicate_rows))

Duplicate state-year rows: 0


In [36]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf


def fit_lagged_fixed_effects(
    panel_df,
    lag_years
):

    diseases = [
        "measles",
        "mumps",
        "pertussis"
    ]

    models = {}
    model_datasets = {}
    results = []

    for disease in diseases:

        coverage_col = (
            f"{disease}_coverage_pct"
        )

        cases_col = (
            f"{disease}_cases"
        )

        # Coverage measured in year t
        coverage_df = (
            panel_df[
                [
                    "state",
                    "year",
                    coverage_col
                ]
            ]
            .rename(
                columns={
                    "year": "coverage_year",
                    coverage_col: "coverage_pct"
                }
            )
            .copy()
        )

        # Required outcome year
        coverage_df["case_year"] = (
            coverage_df["coverage_year"]
            + lag_years
        )

        # Disease cases measured in the outcome year
        cases_df = (
            panel_df[
                [
                    "state",
                    "year",
                    cases_col
                ]
            ]
            .rename(
                columns={
                    "year": "case_year",
                    cases_col: "cases"
                }
            )
            .copy()
        )

        # Match only exact calendar-year pairs
        model_df = coverage_df.merge(
            cases_df,
            on=["state", "case_year"],
            how="inner",
            validate="one_to_one"
        )

        model_df = model_df.dropna(
            subset=[
                "state",
                "coverage_year",
                "case_year",
                "coverage_pct",
                "cases"
            ]
        ).copy()

        model_df = model_df[
            model_df["cases"] >= 0
        ].copy()

        # Confirm the correct lag
        assert (
            model_df["case_year"]
            - model_df["coverage_year"]
            == lag_years
        ).all()

        # Keep states with at least two valid pairs
        model_df["years_available"] = (
            model_df
            .groupby("state")["case_year"]
            .transform("nunique")
        )

        model_df = model_df[
            model_df["years_available"] >= 2
        ].copy()

        model_df["log_cases"] = np.log1p(
            model_df["cases"]
        )

        # State and outcome-year fixed effects
        model = smf.ols(
            """
            log_cases ~ coverage_pct
                      + C(state)
                      + C(case_year)
            """,
            data=model_df
        ).fit(
            cov_type="cluster",
            cov_kwds={
                "groups": model_df["state"]
            }
        )

        coefficient = model.params[
            "coverage_pct"
        ]

        ci_lower, ci_upper = (
            model.conf_int()
            .loc["coverage_pct"]
        )

        models[disease] = model
        model_datasets[disease] = model_df

        years_used = sorted(
            model_df["case_year"].unique()
        )

        results.append({
            "disease": disease.title(),
            "model": f"{lag_years}-year lag",
            "lag_years": lag_years,
            "observations": int(model.nobs),
            "states": model_df["state"].nunique(),
            "years": len(years_used),
            "first_case_year": min(years_used),
            "last_case_year": max(years_used),
            "coverage_coefficient": coefficient,
            "percent_change_cases": (
                100 * (np.exp(coefficient) - 1)
            ),
            "ci_lower_pct": (
                100 * (np.exp(ci_lower) - 1)
            ),
            "ci_upper_pct": (
                100 * (np.exp(ci_upper) - 1)
            ),
            "p_value": model.pvalues[
                "coverage_pct"
            ],
            "r_squared": model.rsquared
        })

    results_df = pd.DataFrame(results)

    return models, model_datasets, results_df

In [37]:
import plotly.graph_objects as go


def plot_fixed_effect_results(
    results_df,
    lag_years,
):  
    title=f"Fixed-Effects Model Results. {lag_years} Lag Year(s)"

    plot_df = results_df.copy()

    # Confidence-interval distances
    plot_df["error_left"] = (
        plot_df["percent_change_cases"]
        - plot_df["ci_lower_pct"]
    )

    plot_df["error_right"] = (
        plot_df["ci_upper_pct"]
        - plot_df["percent_change_cases"]
    )

    colors = {
        "Measles": "#E74C3C",
        "Mumps": "#3498DB",
        "Pertussis": "#2ECC71"
    }

    plot_df["color"] = (
        plot_df["disease"].map(colors)
    )

    plot_df["estimate_label"] = (
        plot_df["percent_change_cases"]
        .map(lambda value: f"{value:+.2f}%")
    )

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=plot_df["percent_change_cases"],
            y=plot_df["disease"],
            mode="markers+text",

            marker={
                "size": 14,
                "color": plot_df["color"]
            },

            text=plot_df["estimate_label"],
            textposition="top center",

            error_x={
                "type": "data",
                "symmetric": False,
                "array": plot_df["error_right"],
                "arrayminus": plot_df["error_left"],
                "thickness": 2,
                "width": 7,
                "color": "#4A4A4A"
            },

            customdata=plot_df[
                [
                    "ci_lower_pct",
                    "ci_upper_pct",
                    "p_value",
                    "observations",
                    "states",
                    "years",
                    "r_squared"
                ]
            ],

            hovertemplate=(
                "<b>%{y}</b><br>"
                "Estimated change: %{x:+.2f}%<br>"
                "95% CI: %{customdata[0]:+.2f}% to "
                "%{customdata[1]:+.2f}%<br>"
                "p-value: %{customdata[2]:.4f}<br>"
                "Observations: %{customdata[3]:.0f}<br>"
                "States: %{customdata[4]:.0f}<br>"
                "Years represented: %{customdata[5]:.0f}<br>"
                "R²: %{customdata[6]:.3f}"
                "<extra></extra>"
            )
        )
    )

    # Zero means no estimated association
    fig.add_vline(
        x=0,
        line_color="black",
        line_dash="dash",
        line_width=1.5,
        annotation_text="No association",
        annotation_position="top"
    )

    fig.update_yaxes(
        categoryorder="array",
        categoryarray=[
            "Pertussis",
            "Mumps",
            "Measles"
        ],
        title_text=None
    )

    fig.update_xaxes(
        title_text=(
            "Estimated change in case burden for a "
            "1-percentage-point increase in coverage"
        ),
        ticksuffix="%"
    )

    fig.update_layout(
        title={
            "text": (
                f"{title}"
                "<br><sup>State and year fixed effects; "
                "horizontal lines show 95% confidence intervals</sup>"
            ),
            "x": 0.5
        },
        template="plotly_white",
        height=500,
        showlegend=False,
        margin={
            "l": 100,
            "r": 60,
            "t": 120,
            "b": 90
        }
    )

    return fig

# Same-year effect model

In [38]:
(same_year_models, same_year_datasets, same_year_df) = fit_lagged_fixed_effects(panel_df=panel_df, lag_years=0)
same_year_df

,disease,model,lag_years,observations,states,years,first_case_year,last_case_year,coverage_coefficient,percent_change_cases,ci_lower_pct,ci_upper_pct,p_value,r_squared
0,Measles,0-year lag,0,316,50,10,1995,2017,-0.024327,-2.403313,-5.363949,0.649945,0.121676,0.556864
1,Mumps,0-year lag,0,620,50,15,1995,2017,0.017327,1.747816,-1.441814,5.040671,0.286307,0.576233
2,Pertussis,0-year lag,0,1151,51,23,1995,2017,0.011236,1.129924,-1.033798,3.340951,0.308569,0.776996


In [39]:
plot_fixed_effect_results(same_year_df, 0)

# 1-year lag effect model

In [40]:
(one_year_models, one_year_datasets, one_year_df) = fit_lagged_fixed_effects(panel_df=panel_df, lag_years=1)
one_year_df

,disease,model,lag_years,observations,states,years,first_case_year,last_case_year,coverage_coefficient,percent_change_cases,ci_lower_pct,ci_upper_pct,p_value,r_squared
0,Measles,1-year lag,1,288,50,9,1996,2017,-0.017050,-1.690511,-5.364385,2.125988,0.380278,0.560841
1,Mumps,1-year lag,1,580,50,14,1996,2017,0.003939,0.394648,-2.125773,2.979973,0.761418,0.570705
2,Pertussis,1-year lag,1,1098,51,22,1996,2017,0.020200,2.040505,-0.171269,4.301284,0.070817,0.778861


In [41]:
plot_fixed_effect_results(one_year_df, 1)

# 2-year lag fixed effect model

In [42]:
(two_year_models, two_year_datasets, two_year_df) = fit_lagged_fixed_effects(panel_df=panel_df, lag_years=2)
two_year_df

,disease,model,lag_years,observations,states,years,first_case_year,last_case_year,coverage_coefficient,percent_change_cases,ci_lower_pct,ci_upper_pct,p_value,r_squared
0,Measles,2-year lag,2,252,49,8,1997,2017,0.001859,0.186042,-4.518677,5.122579,0.939626,0.525478
1,Mumps,2-year lag,2,540,50,13,1997,2017,-0.003504,-0.349783,-3.373295,2.768336,0.823617,0.575090
2,Pertussis,2-year lag,2,1047,51,21,1997,2017,0.009076,0.911779,-1.133136,2.998989,0.384876,0.786138


In [46]:
plot_fixed_effect_results(two_year_df, 2)

In [44]:
(three_year_models, three_year_datasets, three_year_df) = fit_lagged_fixed_effects(panel_df=panel_df, lag_years=3)
three_year_df

,disease,model,lag_years,observations,states,years,first_case_year,last_case_year,coverage_coefficient,percent_change_cases,ci_lower_pct,ci_upper_pct,p_value,r_squared
0,Measles,3-year lag,3,220,49,7,1998,2017,0.003826,0.383340,-4.331276,5.330295,0.876121,0.499876
1,Mumps,3-year lag,3,499,49,12,1998,2017,-0.024452,-2.415570,-5.825797,1.118148,0.177885,0.571014
2,Pertussis,3-year lag,3,995,50,20,1998,2017,0.003374,0.337982,-1.445506,2.153745,0.712325,0.789139


In [47]:
plot_fixed_effect_results(three_year_df, 3)

In [ ]:
(four_year_models, four_year_datasets, four_year_df) = fit_lagged_fixed_effects(panel_df=panel_df, lag_years=4)
four_year_df

,disease,model,lag_years,observations,states,years,first_case_year,last_case_year,coverage_coefficient,percent_change_cases,ci_lower_pct,ci_upper_pct,p_value,r_squared
0,Measles,4-year lag,4,197,49,6,1999,2017,0.008091,0.812394,-5.686412,7.759009,0.811894,0.502615
1,Mumps,4-year lag,4,459,49,11,1999,2017,-0.013561,-1.346993,-4.684078,2.106927,0.439873,0.573848
2,Pertussis,4-year lag,4,945,50,19,1999,2017,0.011758,1.182692,-0.709944,3.111404,0.222305,0.788818


In [ ]:
(five_year_models, five_year_datasets, five_year_df) = fit_lagged_fixed_effects(panel_df=panel_df, lag_years=5)
five_year_df

,disease,model,lag_years,observations,states,years,first_case_year,last_case_year,coverage_coefficient,percent_change_cases,ci_lower_pct,ci_upper_pct,p_value,r_squared
0,Measles,5-year lag,5,162,49,5,2000,2017,-0.014731,-1.462305,-8.726965,6.380568,0.706172,0.564863
1,Mumps,5-year lag,5,417,49,10,2000,2017,0.009021,0.906179,-5.219995,7.428322,0.777719,0.580118
2,Pertussis,5-year lag,5,895,50,18,2000,2017,0.021786,2.202476,0.735770,3.690536,0.003137,0.789405


In [67]:
(year_models, _year_datasets, _year_df) = fit_lagged_fixed_effects(panel_df=panel_df, lag_years=10)
_year_df

,disease,model,lag_years,observations,states,years,first_case_year,last_case_year,coverage_coefficient,percent_change_cases,ci_lower_pct,ci_upper_pct,p_value,r_squared
0,Measles,10-year lag,10,98,49,2,2016,2017,-0.018439,-1.826978,-13.619991,11.576074,0.777642,0.661911
1,Mumps,10-year lag,10,300,49,7,2011,2017,-0.005831,-0.581408,-8.070880,7.518232,0.883985,0.630566
2,Pertussis,10-year lag,10,647,50,13,2005,2017,-0.003763,-0.375640,-1.901570,1.174026,0.632734,0.807843


In [68]:
plot_fixed_effect_results(_year_df, 10)